[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C37_MLOps_Course/02_versioning/02_versioning.ipynb)

# 02 · 从零实现内容寻址版本库

本 notebook 从零造一个「小 DVC / 小 Git」：**内容寻址存储（CAS）+ 不可变快照 + 版本标签 + 血缘 DAG + 多层 diff**，用 **标准库（`hashlib`/`json`/`os`/`tempfile`）+ pandas** 实现，落真盘、过 `assert`。

**路线**：① CAS 的 put/get 与三不变量 → ② manifest 快照 → ③ 标签（指针 vs 真身）+ 回滚 → ④ 血缘 DAG（回溯/影响/查环）→ ⑤ manifest/schema diff → ✏️ 4 道练习 → 📖 答案 → 🧪 真实 CSV 的行级 diff。

In [ ]:
import os, json, hashlib, tempfile
import numpy as np
import pandas as pd
print('环境就绪 ✅ | pandas', pd.__version__)

## 1 · 内容寻址存储：put / get / 三不变量

`put(bytes)->hash` 算哈希并落盘到 `objects/<hash[:2]>/<hash[2:]>`（Git 同款布局）；`get(hash)->bytes` 取回并**自校验完整性**。

三不变量：**确定性**（同内容同哈希）、**完整性**（取回重哈希==地址）、**去重**（同内容只存一份）。

In [ ]:
class CAS:
    def __init__(self, root):
        self.obj = os.path.join(root, 'objects')
        os.makedirs(self.obj, exist_ok=True)

    def _loc(self, h):
        return os.path.join(self.obj, h[:2], h[2:])

    def put(self, data: bytes) -> str:
        h = hashlib.sha256(data).hexdigest()
        loc = self._loc(h)
        if not os.path.exists(loc):                  # 已存在则不重复写（去重）
            os.makedirs(os.path.dirname(loc), exist_ok=True)
            with open(loc, 'wb') as f:
                f.write(data)
        return h

    def get(self, h: str) -> bytes:
        with open(self._loc(h), 'rb') as f:
            data = f.read()
        assert hashlib.sha256(data).hexdigest() == h, '完整性校验失败：内容被损坏！'
        return data

    def exists(self, h):
        return os.path.exists(self._loc(h))

ROOT = tempfile.mkdtemp()
cas = CAS(ROOT)
h1 = cas.put(b'col=age\n23\n31')
h2 = cas.put(b'col=age\n23\n31')          # 同内容
h3 = cas.put(b'col=age\n23\n99')          # 改一个字节
assert h1 == h2, '确定性：同内容必得同哈希'
assert h1 != h3, '内容变一字节，哈希必变'
assert cas.get(h1) == b'col=age\n23\n31', '完整性：取回内容一致'
print('hash =', h1[:16], '... | 落盘于 objects/' + h1[:2] + '/' + h1[2:18] + '...')
print('✅ CAS 三不变量成立：确定性 / 完整性 / 去重')

In [ ]:
# 验证去重确实省了存储：put 同内容 100 次，objects 下只有 2 个文件（h1 与 h3）
for _ in range(100):
    cas.put(b'col=age\n23\n31')
n_objects = sum(len(files) for _,_,files in os.walk(cas.obj))
print('put 了 100+ 次，实际存储对象数 =', n_objects)
assert n_objects == 2, '去重后只应有 2 个不同内容的对象'
print('✅ 去重生效：相同内容只占一份存储')

## 2 · manifest 快照：给一组文件一个版本

数据集 = 多个文件。**manifest** 是「文件名→内容哈希」的字典；对 manifest 自身做规范化哈希，得到整个快照的 **version_id**。

未改动的文件跨版本共享同一个 blob —— 这就是「只存 diff、却能重建任意版本」的秘密。

In [ ]:
def canon_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def snapshot(cas, files: dict) -> tuple:
    '''files: {name: bytes}; 返回 (version_id, manifest{name:hash}).'''
    manifest = {name: cas.put(data) for name, data in files.items()}
    version_id = 'v:' + canon_hash(manifest)[:16]
    return version_id, manifest

v1, m1 = snapshot(cas, {'train.csv': b'a,b\n1,2\n3,4', 'test.csv': b'a,b\n5,6'})
# v2：只改 train.csv，test.csv 不动
v2, m2 = snapshot(cas, {'train.csv': b'a,b\n1,2\n3,9', 'test.csv': b'a,b\n5,6'})
print('version v1 =', v1, '| v2 =', v2)
assert v1 != v2, '改了 train.csv，版本号必变'
assert m1['test.csv'] == m2['test.csv'], 'test.csv 没改，应共享同一 blob（去重）'
assert m1['train.csv'] != m2['train.csv'], 'train.csv 改了，哈希必不同'
# 可从快照完整重建任意版本
rebuilt = cas.get(m1['train.csv'])
assert rebuilt == b'a,b\n1,2\n3,4', 'v1 的 train.csv 可完整重建'
print('✅ manifest 快照：版本随内容变、未改文件共享存储、任意版本可重建')

## 3 · 版本标签：可移动指针 vs 不可变真身 + 秒级回滚

**哈希是不可变真身，标签是可移动指针**。回滚 = 把标签从坏哈希指回好哈希，数据零拷贝。

关键纪律：每次标签移动都记进 **append-only 日志**，这样「当时 prod 指向哪个哈希」可审计。

In [ ]:
class TagStore:
    def __init__(self):
        self.tags = {}          # name -> 当前指向的 version/hash
        self.log = []           # append-only: (tag, old, new)
    def set(self, name, target):
        old = self.tags.get(name)
        self.tags[name] = target
        self.log.append((name, old, target))    # 记录每次移动
    def resolve(self, name):
        return self.tags[name]
    def history(self, name):
        return [(o, n) for (t, o, n) in self.log if t == name]

tags = TagStore()
tags.set('champion', v1)              # 当前生产数据版本
tags.set('champion', v2)              # 晋升到 v2
assert tags.resolve('champion') == v2
# v2 出问题，秒级回滚到 v1（数据一个字节都不用动）
tags.set('champion', v1)
assert tags.resolve('champion') == v1, '回滚确定且即时'
# 审计：champion 标签的完整移动历史都在
hist = tags.history('champion')
print('champion 移动历史 (old->new):')
for o, n in hist:
    print('   ', o, '->', n)
assert len(hist) == 3, '三次移动都被记录'
assert hist[-1] == (v2, v1), '最后一次是 v2->v1 的回滚'
print('✅ 标签可移动、回滚零拷贝、移动历史可审计')

## 4 · 血缘 DAG：回溯、影响分析、查环

lineage = 「由…产生」的有向无环图：节点是资产，边 `child <- parent`。支撑两类查询：
**向上游回溯**（这模型用了哪些数据）与 **向下游影响分析**（这数据坏了，哪些模型要重训）。加边时**检测环**并拒绝。

In [ ]:
class Lineage:
    def __init__(self):
        self.parents = {}     # node -> set(直接上游)
    def add(self, node, parents=()):
        self.parents.setdefault(node, set())
        for p in parents:
            self.parents.setdefault(p, set())
            # 查环：若 node 是 p 的祖先，则加这条边会成环
            assert node not in self.ancestors(p) and node != p, f'加边 {p}->{node} 会成环！'
            self.parents[node].add(p)
    def ancestors(self, node):
        seen = set(); stack = list(self.parents.get(node, ()))
        while stack:
            x = stack.pop()
            if x not in seen:
                seen.add(x); stack += list(self.parents.get(x, ()))
        return seen
    def descendants(self, node):
        out = set()
        for n in self.parents:
            if node in self.ancestors(n):
                out.add(n)
        return out

lin = Lineage()
lin.add('data:raw@v1')
lin.add('data:clean@v2', ['data:raw@v1', 'code:clean@c1'])
lin.add('model:clf@m3', ['data:clean@v2', 'code:train@c2'])
lin.add('eval:report@e1', ['model:clf@m3'])

# 回溯：model:clf@m3 的全部上游
anc = lin.ancestors('model:clf@m3')
print('model:clf@m3 的祖先 =', sorted(anc))
assert 'data:raw@v1' in anc and 'code:clean@c1' in anc, '传递回溯到最上游'
# 影响分析：data:raw@v1 坏了，哪些下游受影响？
desc = lin.descendants('data:raw@v1')
print('data:raw@v1 坏了，受影响的下游 =', sorted(desc))
assert 'model:clf@m3' in desc and 'eval:report@e1' in desc, '影响传递到最下游'
print('✅ 血缘回溯 + 影响分析正确')

In [ ]:
# 查环：尝试加一条会成环的边，应被拒绝
raised = False
try:
    lin.add('data:raw@v1', ['eval:report@e1'])   # raw 依赖它的下游 -> 环
except AssertionError as e:
    raised = True; print('成环被拒：', e)
assert raised, 'DAG 必须拒绝成环的边'
print('✅ 无环约束生效（DAG 保证可拓扑排序、回溯必终止）')

## 5 · diff：manifest 层与 schema 层

回滚前要知道「两版差在哪」。**manifest diff** 只比哈希（免读内容，最便宜）：找出新增/删除/修改/未变的文件。

**schema diff** 比列结构（列增删、类型变化）——schema 变化常比数据漂移更危险。

In [ ]:
def manifest_diff(m_old, m_new):
    keys = set(m_old) | set(m_new)
    out = {'added': [], 'removed': [], 'modified': [], 'unchanged': []}
    for k in sorted(keys):
        if k not in m_old:        out['added'].append(k)
        elif k not in m_new:      out['removed'].append(k)
        elif m_old[k] != m_new[k]: out['modified'].append(k)    # 只比哈希！
        else:                      out['unchanged'].append(k)
    return out

# 在 m1, m2 之上再造一个 m3（加一个文件、删一个文件）
v3, m3 = snapshot(cas, {'train.csv': m2 and b'a,b\n1,2\n3,9', 'schema.json': b'{}'})
# 用真实哈希对齐：m3 的 train.csv 与 m2 相同内容
d = manifest_diff(m2, m3)
print('m2 -> m3 diff:', {k: v for k, v in d.items() if v})
assert 'schema.json' in d['added'], 'm3 新增了 schema.json'
assert 'test.csv' in d['removed'], 'm3 删除了 test.csv'
assert 'train.csv' in d['unchanged'], 'train.csv 同内容 -> 免读内容即判未变'
print('✅ manifest diff 正确（修改判定只比哈希，对 GB 级数据也是 O(文件数)）')

In [ ]:
def schema_diff(df_old, df_new):
    cols_old, cols_new = set(df_old.columns), set(df_new.columns)
    out = {'added_cols': sorted(cols_new - cols_old),
           'removed_cols': sorted(cols_old - cols_new),
           'type_changed': []}
    for c0 in cols_old & cols_new:
        if str(df_old[c0].dtype) != str(df_new[c0].dtype):
            out['type_changed'].append((c0, str(df_old[c0].dtype), str(df_new[c0].dtype)))
    return out

df_a = pd.DataFrame({'age': [23, 31], 'city': ['x', 'y']})
df_b = pd.DataFrame({'age': ['23', '31'], 'zip': [1, 2]})   # age 变字符串、city->zip
sd = schema_diff(df_a, df_b)
print('schema diff:', sd)
assert sd['added_cols'] == ['zip'] and sd['removed_cols'] == ['city']
assert sd['type_changed'] and sd['type_changed'][0][0] == 'age'
print('✅ schema diff 抓住列增删 + 类型变化（上游改格式的早期预警）')

---
## ✏️ 练习 1：验证 CAS 的完整性自校验

CAS 的 `get` 承诺「取回内容重哈希 == 地址」。实现 `verify_store(cas, h)`：取回 `h` 对应内容、重新计算哈希、返回是否 `== h`。再实现 `detect_corruption(cas, h)`：模拟存储损坏（篡改盘上的字节）后，`get` 应当抛 `AssertionError`——返回是否成功检测到损坏。

In [ ]:
def verify_store(cas, h):
    # TODO: 用 cas.get(h) 取回内容，重算 sha256，返回是否等于 h（True/False）
    raise NotImplementedError

def detect_corruption(cas, h):
    # TODO: 直接打开 cas._loc(h) 往里追加一个字节(模拟损坏)，再调 cas.get(h)；
    #       若抛 AssertionError 说明检测到损坏，返回 True；否则 False。
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
cas_t = CAS(tempfile.mkdtemp())
ht = cas_t.put(b'hello world')
assert verify_store(cas_t, ht) == True, '未损坏时应校验通过'
assert detect_corruption(cas_t, ht) == True, '损坏后 get 应抛错被检测到'
print('✅ 练习 1 通过：内容寻址的完整性自校验有效')

## ✏️ 练习 2：增量存储省了多少

内容寻址的卖点是「未改动文件不重复存」。实现 `storage_saved(cas, m_old, m_new)`：返回 v_new 相对 v_old **新增的对象数**（即 m_new 里、哈希不在 m_old 值集合中的文件数）——这就是这次版本实际多占的存储块数。

In [ ]:
def storage_saved(cas, m_old, m_new):
    # TODO: 返回 m_new 中哈希不出现在 m_old.values() 里的文件个数（=本次真正新增的对象数）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
casx = CAS(tempfile.mkdtemp())
_, ma = snapshot(casx, {'a': b'AAA', 'b': b'BBB', 'c': b'CCC'})
_, mb = snapshot(casx, {'a': b'AAA', 'b': b'BBB', 'c': b'ZZZ'})  # 只改 c
newcnt = storage_saved(casx, ma, mb)
assert newcnt == 1, f'只改了 c，应只新增 1 个对象，得到 {newcnt}'
_, mc = snapshot(casx, {'a': b'AAA', 'b': b'BBB', 'c': b'CCC'})  # 与 ma 完全相同
assert storage_saved(casx, ma, mc) == 0, '完全相同的版本应新增 0 个对象'
print('✅ 练习 2 通过：能算出增量存储成本')

## ✏️ 练习 3：找出两个资产的最近公共祖先（共同上游）

排查「两个模型为何都出问题」时，常想知道它们**共享哪些上游**。实现 `common_ancestors(lin, a, b)`：
返回 `a` 与 `b` 的祖先集合的 **交集**（含它们共享的全部上游资产）。

In [ ]:
def common_ancestors(lin, a, b):
    # TODO: 返回 lin.ancestors(a) 与 lin.ancestors(b) 的交集（集合）
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
L = Lineage()
L.add('raw')
L.add('clean', ['raw'])
L.add('m_A', ['clean'])
L.add('m_B', ['clean'])
L.add('m_C', ['raw'])           # 走另一条路
comm = common_ancestors(L, 'm_A', 'm_B')
assert comm == {'raw', 'clean'}, f'm_A 与 m_B 共享 raw+clean，得到 {comm}'
assert common_ancestors(L, 'm_A', 'm_C') == {'raw'}, 'm_A 与 m_C 只共享 raw'
print('✅ 练习 3 通过：能定位两个资产的共同上游')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def verify_store(cas, h):
    data = cas.get(h)
    return hashlib.sha256(data).hexdigest() == h

def detect_corruption(cas, h):
    with open(cas._loc(h), 'ab') as f:
        f.write(b'X')                       # 篡改盘上内容
    try:
        cas.get(h)
        return False
    except AssertionError:
        return True

In [ ]:
# 练习 2 参考答案
def storage_saved(cas, m_old, m_new):
    old_hashes = set(m_old.values())
    return sum(1 for h in m_new.values() if h not in old_hashes)

In [ ]:
# 练习 3 参考答案
def common_ancestors(lin, a, b):
    return lin.ancestors(a) & lin.ancestors(b)

---
## 🧪 真实数据胶囊：真实 CSV 的行级 diff

manifest diff 告诉你「哪个文件变了」，但有时你要「**具体哪几行变了**」。

用 pandas 对两版真实 CSV 做 **行级 diff**：按主键对齐，找出新增行、删除行、被修改的行。这是数据审查与「为什么模型突然变差」溯源的利器。

In [ ]:
# 两版真实数据（v1 -> v2：改了 1 行、加了 1 行、删了 1 行）
import io
csv_v1 = 'id,age,income\n1,25,50000\n2,40,80000\n3,33,60000\n4,52,95000'
csv_v2 = 'id,age,income\n1,25,50000\n2,40,82000\n4,52,95000\n5,29,55000'
df1 = pd.read_csv(io.StringIO(csv_v1)).set_index('id')
df2 = pd.read_csv(io.StringIO(csv_v2)).set_index('id')

def row_diff(df_old, df_new):
    old_ids, new_ids = set(df_old.index), set(df_new.index)
    added = sorted(new_ids - old_ids)
    removed = sorted(old_ids - new_ids)
    modified = []
    for i in sorted(old_ids & new_ids):
        if not df_old.loc[i].equals(df_new.loc[i]):
            modified.append(i)
    return {'added': added, 'removed': removed, 'modified': modified}

rd = row_diff(df1, df2)
print('行级 diff:', rd)
assert rd['added'] == [5], 'id=5 是新增行'
assert rd['removed'] == [3], 'id=3 被删除'
assert rd['modified'] == [2], 'id=2 的 income 被改 (80000->82000)'
print('✅ 行级 diff 跑通：精确定位新增/删除/修改的行')

**🧪 胶囊练习**：实现 `changed_columns(df_old, df_new, row_id)`：对一个**被修改**的行，返回 **具体哪些列的值变了** 的列表 `[(列名, 旧值, 新值), ...]`。

In [ ]:
def changed_columns(df_old, df_new, row_id):
    # TODO: 对 row_id 这一行，逐列比较 df_old 与 df_new 的值，
    #       返回 [(col, old_val, new_val), ...]（只含值不同的列）
    raise NotImplementedError

In [ ]:
# 自测
ch = changed_columns(df1, df2, 2)
assert len(ch) == 1 and ch[0][0] == 'income', '只有 income 列变了'
assert ch[0][1] == 80000 and ch[0][2] == 82000, '旧值 80000 -> 新值 82000'
print('id=2 的变化:', ch)
print('✅ 胶囊练习通过：下钻到列级变化')

In [ ]:
# 📖 胶囊参考答案
def changed_columns(df_old, df_new, row_id):
    out = []
    for col in df_old.columns:
        ov, nv = df_old.loc[row_id, col], df_new.loc[row_id, col]
        if ov != nv:
            out.append((col, ov, nv))
    return out

---
## 🔧 旁注：这套东西对应 DVC / Git 的什么

你刚写的版本库几乎逐一对应 DVC 与 Git 的内核（伪代码，**本环境不跑**）：

```bash
dvc add data/train.csv        # == 我们的 cas.put + 写 manifest（.dvc 文件存哈希指针）
git commit -m 'data v2'       # == snapshot：把 manifest 哈希进 commit（version_id）
git tag v1.2 / git checkout prod  # == TagStore.set / resolve（指针 vs 真身）
dvc repro                     # == 沿 lineage DAG 拓扑重放流水线
git diff v1 v2                # == manifest_diff / row_diff
```

对应关系：`cas.put`↔DVC cache（内容寻址去重）、`manifest`↔`.dvc`/Git tree、`TagStore`↔Git branch/tag、`Lineage`↔DVC pipeline DAG。DVC/Git 多出来的是远程存储、网络传输、压缩、并发——但**内容寻址的内核与你写的一模一样**。

### 小结
- **内容寻址（CAS）**：用内容哈希当地址 -> 不可变 + 去重 + 完整性自校验，是版本系统的地基。
- **manifest 快照**：一组「文件名→哈希」，对它哈希得 version_id；未改文件跨版本共享 blob。
- **标签是可移动指针，哈希是不可变真身**：回滚=移指针（零拷贝、确定）；审计要记当时指向的哈希。
- **血缘 DAG**：回溯上游 + 下游影响分析 + 无环约束；**diff 三层**（manifest→schema→行级，由粗到细）。

下一站：**模块 03 · 评测门禁与 CI/CD** —— 这个新模型该不该放行？它真比线上那个好，还是只是噪声？